# 🌀 **Binary Accretion Emulator** — *Explore Accretion Variability in Real Time*

Welcome to the interactive emulator for **accretion variability from circumbinary disks**!  
Here, you can explore how the mass accretion rates onto binaries depend on:

- 🔵 **Binary eccentricity** ($e_{\rm b}$)
- ⚖️ **Binary mass ratio** ($q_{\rm b}$)

Using the sliders below, choose values for $e_{\rm b}$ and $q_{\rm b}$ and watch the accretion rate ($\dot{M}$) time series change in real time.

--- 

## 🌌 **What does this represent?**

Binary systems embedded in circumbinary disks produce complex and beautiful accretion variability.  
As gas streams penetrate the cavity and interact with the binary, they create:

- **Periodic pulses**
- **Chaotic bursts**
- **Quasi-steady accretion regimes**

The patterns depend sensitively on $e_{\rm b}$ and $q_{\rm b}$ — now you can visualize them interactively!

---

## 🎮 **Instructions**

- Use the sliders to adjust **$e_{\rm b}$** and **$q_{\rm b}$**
- The plot will automatically update to show how the accretion pattern responds
- Try extreme values to see how things change dramatically!

---

## 🚀 **Try me:**

| Parameter | Meaning | Range |
|-----------|---------|-------|
| $e_{\rm b}$ | Binary eccentricity | 0.0 → 0.8 |
| $q_{\rm b}$ | Mass ratio (secondary/primary) | 0.1 → 1.0 |

Have fun exploring the physics of accretion variability in binaries!

---

> _"Where gas flows, chaos and beauty emerge."_ 🌊✨

In [5]:
%matplotlib qt5

import sys
from functools import lru_cache
from pathlib import Path

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'pyproject.toml').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, CheckButtons, Button
from matplotlib.patches import Patch

import os
print(bool(os.environ.get("ZENODO_SNDBX_TOKEN")))


from demo_constants import Pb
import calypso
from calypso.data.loader import load_single_component

plt.rcParams.update({'xtick.labelsize': 14, 'ytick.labelsize': 14})

color_pred_Mb = '#1f77b4'
color_pred_M1 = '#2ca02c'
color_pred_M2 = '#d62728'
color_ref_Mb = '#1f5a85'
color_ref_M1 = '#2f7d32'
color_ref_M2 = '#b23a48'
lw_true = 1.2
alpha_true = 0.9
std_alpha = 0.28

show_ts0 = True
show_stats0 = False
eb0, qb0 = 0.7, 1.0
nwindows, norb, nbins = 500, 10, 100
log_model = True
match_tolerance = 0.02
sample_seed0 = 0
current_sample_seed = sample_seed0
stats_sample_count0 = 64

def load_ts(nwindows, norb, nbins, log_model):
    loaded_ts = {'train': {'Mb': {}, 'M1': {}, 'M2': {}}, 'test': {'Mb': {}, 'M1': {}, 'M2': {}}}
    for split in ('train', 'test'):
        for component in ('Mb', 'M1', 'M2'):
            X, y = load_single_component(split, component, nwindows, norb, nbins, log_model)
            for i in range(len(X)):
                eb_val, qb_val = X[i].tolist()
                key = (round(eb_val, 2), round(qb_val, 2))
                loaded_ts[split][component].setdefault(key, []).append(y[i])
    return loaded_ts

loaded_TS = load_ts(nwindows, norb, nbins, log_model)
emulator = calypso.load_emulator()
time = np.linspace(0, norb * Pb, nbins * norb)
t = time / Pb

def find_true_series_candidates(loaded_ts, split, component, eb, qb):
    for (ebv, qbv), arrs in loaded_ts[split][component].items():
        if np.isclose(ebv, eb, atol=match_tolerance) and np.isclose(qbv, qb, atol=match_tolerance):
            return [10**arr if log_model else arr for arr in arrs]
    return None

def select_reference_bundle(loaded_ts, eb, qb, sample_seed):
    rng = np.random.default_rng(sample_seed)
    for split in ('test', 'train'):
        candidates = {comp: find_true_series_candidates(loaded_ts, split, comp, eb, qb) for comp in ('Mb', 'M1', 'M2')}
        if all(candidates[comp] is not None and len(candidates[comp]) > 0 for comp in ('Mb', 'M1', 'M2')):
            n_available = min(len(candidates['Mb']), len(candidates['M1']), len(candidates['M2']))
            idx = int(rng.integers(n_available))
            return {
                'Mb': candidates['Mb'][idx],
                'M1': candidates['M1'][idx],
                'M2': candidates['M2'][idx],
                'split': split,
                'index': idx,
            }
    return {'Mb': None, 'M1': None, 'M2': None, 'split': None, 'index': None}

def get_prediction_samples(eb, qb, n_samples, sample_seed):
    pred = emulator.predict(eb, qb, n_samples=n_samples, rng=np.random.default_rng(sample_seed))
    if log_model:
        return pred['Mb'], pred['M1'], pred['M2']
    return pred['Mb'], pred['M1'], pred['M2']

def summarize_component(samples):
    samples = np.asarray(samples, dtype=float)
    if log_model:
        log_mean = samples.mean(axis=0)
        log_std = samples.std(axis=0)
        center = 10**log_mean
        lower = 10**(log_mean - log_std)
        upper = 10**(log_mean + log_std)
        single = 10**samples[0]
    else:
        center = samples.mean(axis=0)
        linear_std = samples.std(axis=0)
        lower = np.clip(center - linear_std, 0.0, None)
        upper = center + linear_std
        single = samples[0]
    return single, center, lower, upper

def get_display_curves(eb, qb, show_stats, sample_seed=current_sample_seed, stats_sample_count=stats_sample_count0):
    sample_count = stats_sample_count if show_stats else 1
    mb_samples, m1_samples, m2_samples = get_prediction_samples(eb, qb, sample_count, sample_seed)
    mb_single, mb_center, mb_lower, mb_upper = summarize_component(mb_samples)
    m1_single, m1_center, m1_lower, m1_upper = summarize_component(m1_samples)
    m2_single, m2_center, m2_lower, m2_upper = summarize_component(m2_samples)
    if show_stats:
        curves = {
            'Mb': mb_center,
            'M1': m1_center,
            'M2': m2_center,
            'Mb_lower': mb_lower,
            'M1_lower': m1_lower,
            'M2_lower': m2_lower,
            'Mb_upper': mb_upper,
            'M1_upper': m1_upper,
            'M2_upper': m2_upper,
            'mode_label': rf'mean $\pm$ 1$\sigma$ | {sample_count} samples',
        }
    else:
        curves = {
            'Mb': mb_single,
            'M1': m1_single,
            'M2': m2_single,
            'Mb_lower': mb_single,
            'M1_lower': m1_single,
            'M2_lower': m2_single,
            'Mb_upper': mb_single,
            'M1_upper': m1_single,
            'M2_upper': m2_single,
            'mode_label': 'single stochastic sample',
        }
    true_ts = select_reference_bundle(loaded_TS, eb, qb, sample_seed)
    return curves, true_ts

def get_xlabels(time_axis):
    n_labels = 5
    dt = (time_axis[-1] - time_axis[0]) / n_labels
    xlabels = [time_axis[0] + i * dt for i in range(n_labels + 1)]
    xlabels_string = [r'$t_0 + $' + f'{i * dt:.1f}' + r'$\,P_{\rm b}$' for i in range(n_labels + 1)]
    return xlabels, xlabels_string

def refresh_std_band(ax, patch, x, lower, upper, color, visible):
    if patch is not None:
        patch.remove()
    if not visible:
        return ax.fill_between(x, lower, upper, color=color, alpha=0.0, zorder=1)
    return ax.fill_between(x, lower, upper, color=color, alpha=std_alpha, zorder=1)

def positive_ylim(*series, pad_frac=0.05, floor=1e-6):
    arrays = []
    for arr in series:
        arr = np.asarray(arr, dtype=float)
        valid = arr[np.isfinite(arr)]
        if valid.size:
            arrays.append(valid)
    if not arrays:
        return floor, 1.0
    values = np.concatenate(arrays)
    ymax = values.max()
    positive = values[values > 0]
    ymin = positive.min() if positive.size else floor
    yrange = max(ymax - ymin, ymax * pad_frac, floor)
    lower = max(floor, ymin - pad_frac * yrange)
    upper = ymax + pad_frac * yrange
    if upper <= lower:
        upper = lower * (1 + pad_frac)
    return lower, upper

def apply_mode_style(show_stats):
    linewidth = 2.4 if show_stats else 2.0
    lMb.set_linewidth(linewidth)
    lM1.set_linewidth(linewidth)
    lM2.set_linewidth(linewidth)
    for line in [lMb, lM1, lM2]:
        line.set_linestyle('-')

def format_ref_label(component, split):
    return f'ref $\dot{{M}}_{{{component}}}$ ({split})'

def update_true_ts_visibility(show_ts):
    for line in reference_lines:
        line.set_visible(show_ts and not np.all(np.isnan(line.get_ydata())))

def update_legends(show_ts, show_stats, ref_labels):
    top_handles = [lMb]
    top_labels = [r'$\dot{M}_{\rm b}$']
    bottom_handles = [lM1, lM2]
    bottom_labels = [r'$\dot{M}_1$', r'$\dot{M}_2$']
    if show_ts and ref_labels['Mb'] is not None and lRefMb.get_visible():
        top_handles.append(lRefMb)
        top_labels.append(format_ref_label('b', ref_labels['Mb']))
    if show_ts and ref_labels['M1'] is not None and lRefM1.get_visible():
        bottom_handles.append(lRefM1)
        bottom_labels.append(format_ref_label('1', ref_labels['M1']))
    if show_ts and ref_labels['M2'] is not None and lRefM2.get_visible():
        bottom_handles.append(lRefM2)
        bottom_labels.append(format_ref_label('2', ref_labels['M2']))
    if show_stats:
        std_patch = Patch(facecolor='gray', edgecolor='none', alpha=std_alpha)
        top_handles.append(std_patch)
        top_labels.append(r'1$\sigma$ envelope')
        bottom_handles.append(std_patch)
        bottom_labels.append(r'1$\sigma$ envelope')
    ax1.legend(top_handles, top_labels, fontsize=14, loc='upper right')
    ax2.legend(bottom_handles, bottom_labels, fontsize=14, loc='upper right')

def position_mode_controls(fig, anchor_ax, toggle_ax, button_ax, toggle_width=0.22, toggle_height=0.06, button_width=0.12, button_height=0.05, margin=0.02, gap=0.015):
    fig.canvas.draw()
    ax_bbox = anchor_ax.get_position()
    renderer = fig.canvas.get_renderer()
    title_bbox = anchor_ax.title.get_window_extent(renderer=renderer).transformed(fig.transFigure.inverted())
    bottom = min(0.98 - toggle_height, max(ax_bbox.y1, title_bbox.y1) + margin)
    left = ax_bbox.x0
    toggle_ax.set_position([left, bottom, toggle_width, toggle_height])
    button_bottom = bottom + 0.5 * (toggle_height - button_height)
    button_ax.set_position([left + toggle_width + gap, button_bottom, button_width, button_height])

curves0, true_ts0 = get_display_curves(eb0, qb0, show_stats0, sample_seed=current_sample_seed)

fig = plt.figure(figsize=(12, 8))
fig.subplots_adjust(top=0.84)
gs = fig.add_gridspec(7, 1, height_ratios=[2, 25, 5, 25, 5, 3, 3], hspace=0.1)

ax1 = fig.add_subplot(gs[1])
lMb, = ax1.plot(t, curves0['Mb'], lw=2.0, color=color_pred_Mb, zorder=2)
mb_std_band = refresh_std_band(ax1, None, t, curves0['Mb_lower'], curves0['Mb_upper'], color_pred_Mb, show_stats0)
mb_ref0, mb_ref_label0 = true_ts0['Mb'], true_ts0['split']
lRefMb, = ax1.plot(t, mb_ref0 if mb_ref0 is not None else np.full_like(t, np.nan), lw=lw_true, alpha=alpha_true, color=color_ref_Mb, linestyle='-', zorder=3)
ax1.set_ylabel(r'$\dot{M}_{\rm b}$', fontsize=16)
ax1.set_title(rf"$\dot{{M}}_{{\rm b}}$ accretion rate, $e_{{\rm b}}$={eb0:.2f}, $q_{{\rm b}}$={qb0:.2f} | {curves0['mode_label']}", fontsize=18)
ax1.grid(alpha=0.3, which='both', linestyle='--')
plt.setp(ax1.get_xticklabels(), visible=False)

ax2 = fig.add_subplot(gs[3], sharex=ax1)
lM1, = ax2.plot(t, curves0['M1'], lw=2.0, color=color_pred_M1, zorder=2)
lM2, = ax2.plot(t, curves0['M2'], lw=2.0, color=color_pred_M2, zorder=2)
m1_std_band = refresh_std_band(ax2, None, t, curves0['M1_lower'], curves0['M1_upper'], color_pred_M1, show_stats0)
m2_std_band = refresh_std_band(ax2, None, t, curves0['M2_lower'], curves0['M2_upper'], color_pred_M2, show_stats0)
m1_ref0, m1_ref_label0 = true_ts0['M1'], true_ts0['split']
m2_ref0, m2_ref_label0 = true_ts0['M2'], true_ts0['split']
lRefM1, = ax2.plot(t, m1_ref0 if m1_ref0 is not None else np.full_like(t, np.nan), lw=lw_true, alpha=alpha_true, color=color_ref_M1, linestyle='-', zorder=3)
lRefM2, = ax2.plot(t, m2_ref0 if m2_ref0 is not None else np.full_like(t, np.nan), lw=lw_true, alpha=alpha_true, color=color_ref_M2, linestyle='-', zorder=3)
ax2.set_ylabel(r'$\dot{M}$', fontsize=16)
ax2.set_xlabel(r'Time / $P_{\rm b}$', fontsize=16)
ax2.set_title(rf"Accretion rates of $\dot{{M}}_1$ and $\dot{{M}}_2$ | {curves0['mode_label']}", fontsize=16)
ax2.grid(alpha=0.3, which='both', linestyle='--')

xlabels, xlabels_string = get_xlabels(t)
ax2.set_xticks(xlabels)
ax2.set_xticklabels(xlabels_string)

reference_lines = [lRefMb, lRefM1, lRefM2]
ref_labels = {'Mb': mb_ref_label0, 'M1': m1_ref_label0, 'M2': m2_ref_label0}
update_true_ts_visibility(show_ts0)
apply_mode_style(show_stats0)
update_legends(show_ts0, show_stats0, ref_labels)

ax1.set_ylim(*positive_ylim(curves0['Mb_lower'], curves0['Mb_upper']))
ax2.set_ylim(*positive_ylim(curves0['M1_lower'], curves0['M1_upper'], curves0['M2_lower'], curves0['M2_upper']))

ax_mode = fig.add_axes([0.07, 0.88, 0.22, 0.06])
mode_toggle = CheckButtons(ax_mode, ['Show true TS', r'mean $\pm$ std'], [show_ts0, show_stats0])
for label in mode_toggle.labels:
    label.set_fontsize(11)
ax_mode.set_title('Display mode', fontsize=11)
ax_resample = fig.add_axes([0.31, 0.88, 0.12, 0.05])
resample_button = Button(ax_resample, 'resample')
position_mode_controls(fig, ax1, ax_mode, ax_resample)

ax_eb = fig.add_subplot(gs[5])
s_eb = Slider(ax_eb, 'e$_b$', 0.0, 0.8, valinit=eb0, valstep=0.01)
ax_qb = fig.add_subplot(gs[6])
s_qb = Slider(ax_qb, 'q$_b$', 0.1, 1.0, valinit=qb0, valstep=0.01)

def update(val=None):
    global current_sample_seed, mb_std_band, m1_std_band, m2_std_band
    eb_val, qb_val = s_eb.val, s_qb.val
    show_ts, show_stats = mode_toggle.get_status()
    curves, true_ts = get_display_curves(eb_val, qb_val, show_stats, sample_seed=current_sample_seed)
    lMb.set_ydata(curves['Mb'])
    lM1.set_ydata(curves['M1'])
    lM2.set_ydata(curves['M2'])
    mb_std_band = refresh_std_band(ax1, mb_std_band, t, curves['Mb_lower'], curves['Mb_upper'], color_pred_Mb, show_stats)
    m1_std_band = refresh_std_band(ax2, m1_std_band, t, curves['M1_lower'], curves['M1_upper'], color_pred_M1, show_stats)
    m2_std_band = refresh_std_band(ax2, m2_std_band, t, curves['M2_lower'], curves['M2_upper'], color_pred_M2, show_stats)
    mb_ref, mb_ref_label = true_ts['Mb'], true_ts['split']
    m1_ref, m1_ref_label = true_ts['M1'], true_ts['split']
    m2_ref, m2_ref_label = true_ts['M2'], true_ts['split']
    lRefMb.set_ydata(mb_ref if mb_ref is not None else np.full_like(t, np.nan))
    lRefM1.set_ydata(m1_ref if m1_ref is not None else np.full_like(t, np.nan))
    lRefM2.set_ydata(m2_ref if m2_ref is not None else np.full_like(t, np.nan))
    ref_labels = {'Mb': mb_ref_label, 'M1': m1_ref_label, 'M2': m2_ref_label}
    update_true_ts_visibility(show_ts)
    apply_mode_style(show_stats)
    update_legends(show_ts, show_stats, ref_labels)
    ax1.set_ylim(*positive_ylim(curves['Mb_lower'], curves['Mb_upper']))
    ax2.set_ylim(*positive_ylim(curves['M1_lower'], curves['M1_upper'], curves['M2_lower'], curves['M2_upper']))
    ax1.set_title(rf"$\dot{{M}}_{{\rm b}}$ accretion rate, $e_{{\rm b}}$={eb_val:.2f}, $q_{{\rm b}}$={qb_val:.2f} | {curves['mode_label']}", fontsize=18)
    ax2.set_title(rf"Accretion rates of $\dot{{M}}_1$ and $\dot{{M}}_2$ | {curves['mode_label']}", fontsize=16)
    position_mode_controls(fig, ax1, ax_mode, ax_resample)
    fig.canvas.draw_idle()

def resample(event):
    global current_sample_seed
    current_sample_seed += 1
    update()

s_eb.on_changed(update)
s_qb.on_changed(update)
mode_toggle.on_clicked(update)
resample_button.on_clicked(resample)

plt.show()


<>:186: SyntaxWarning: invalid escape sequence '\d'
<>:186: SyntaxWarning: invalid escape sequence '\d'
/var/folders/4d/c_wdwwqx2lq38jg5lnb4j1l00000gn/T/ipykernel_39646/1181126440.py:186: SyntaxWarning: invalid escape sequence '\d'
  return f'ref $\dot{{M}}_{{{component}}}$ ({split})'


False
Cached manifest is up to date.
Cached manifest is up to date.
